# Ejercicio 2: Implementación de la Distancia de Gower

El script debe contener una implementación de la Distancia de Gower:

1. Implementar la fórmula de Gower.
2. Validar con un subconjunto pequeño (ej. 5 filas).
3. Probar el cálculo de la matriz de distancias en una muestra pequeña del dataset.

In [2]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from IPython.display import display

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# =================================================================
# Carga del DataFrame Original (con valores faltantes)
# =================================================================
# Obtener el dataset "Adult" (id=2)
adult = fetch_ucirepo(id=2)

# Data como DataFrames de pandas
X = adult.data.features
y = adult.data.targets
df = pd.concat([X, y], axis=1)

# Reemplazar el caracter '?' por NaN de NumPy para un manejo estándar
df.replace('?', np.nan, inplace=True)

# =================================================================
# Implementación Vectorizada de la Distancia de Gower
# =================================================================

# 1. Preparación de los datos para Gower
# -----------------------------------------------------------------
# Tomamos una muestra de 5 filas específicas
sample_indices = [27, 30, 31, 32, 33]
df_sample = df.loc[sample_indices].copy()

print(f"--- Se ha tomado una muestra de {len(df_sample)} filas (índices: {sample_indices}) ---")

# Identificar tipos de variables
numeric_cols = ['age', 'fnlwgt', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# --- Manejo de la Variable Ordinal 'education' ---
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}


# 2. Cálculo Vectorizado de la Matriz de Distancias
# -----------------------------------------------------------------
print("\nCalculando la matriz de distancias (VERSIÓN VECTORIZADA)...\n")

# Paso A: Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Paso B: Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Paso C: Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Paso D: Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Paso E: Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix_vectorized = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)


# 3. Visualización de Resultados
# -----------------------------------------------------------------
gower_matrix_df = pd.DataFrame(gower_matrix_vectorized, 
                               index=df_sample.index, 
                               columns=df_sample.index)

print("--- Validación de la Distancia de Gower con Variables Ordinales ---")
print("Matriz de Distancias de Gower resultante (versión vectorizada):\n")
display(gower_matrix_df)

KeyboardInterrupt: 